# MapAid — Entrenamiento del modelo de detección de daños

**U-Net (ResNet34) entrenado con xBD — Pieza 1 de la IA**

Este notebook entrena el modelo que detecta edificios dañados comparando
imágenes de satélite antes y después de un desastre.

**Dataset:** `qianlanzz/xbd-dataset` (tier1 + tier3)  
**Tiempo estimado:** ~3-4 horas con GPU P100  
**Resultado:** `modelo_danos.pt` — copiar en `backend/ia/`

---

### Antes de ejecutar
1. Añadir el dataset: panel derecho → **Add data** → buscar `qianlanzz/xbd-dataset`
2. Activar GPU: **Session options** → **GPU P100**
3. Ejecutar todas las celdas en orden


## 1. Verificar GPU

In [ ]:
import torch

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {dispositivo}")

if dispositivo.type == "cuda":
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Sin GPU — activa GPU P100 en Session options")


## 2. Instalar dependencias

In [ ]:
!pip install -q segmentation-models-pytorch albumentations
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
print("✅ Dependencias OK")


## 3. Preparar el dataset (tier1 + tier3)

In [ ]:
import json, re
from pathlib import Path
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

XBD_IMAGES_T1 = Path("/kaggle/input/xbd-dataset/xbd/tier1/images")
XBD_LABELS_T1 = Path("/kaggle/input/xbd-dataset/xbd/tier1/labels")
XBD_IMAGES_T3 = Path("/kaggle/input/xbd-dataset/xbd/tier3/images")
XBD_LABELS_T3 = Path("/kaggle/input/xbd-dataset/xbd/tier3/labels")

DANO_A_CLASE = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3,
    "un-classified": 4,
}

TAMANO = 512
N_CLASES = 5


def wkt_a_mascara(wkt, ancho, alto):
    numeros = re.findall(r"-?\d+\.?\d*", wkt)
    if not numeros:
        return np.zeros((alto, ancho), dtype=np.uint8)
    arr = np.array(numeros, dtype=float)
    if arr.size % 2 != 0:
        arr = arr[:-(arr.size % 2)]
    if arr.size < 6:
        return np.zeros((alto, ancho), dtype=np.uint8)
    pts = arr.reshape(-1, 2).astype(np.int32)
    mascara = np.zeros((alto, ancho), dtype=np.uint8)
    cv2.fillPoly(mascara, [pts], 1)
    return mascara


class XBDDataset(Dataset):
    def __init__(self, escenas, transformacion=None):
        self.escenas = escenas
        self.transformacion = transformacion

    def __len__(self):
        return len(self.escenas)

    def __getitem__(self, idx):
        base, img_dir, lbl_dir = self.escenas[idx]
        img_pre  = cv2.imread(str(img_dir / f"{base}_pre_disaster.png"))
        img_post = cv2.imread(str(img_dir / f"{base}_post_disaster.png"))
        img_pre  = cv2.cvtColor(img_pre,  cv2.COLOR_BGR2RGB)
        img_post = cv2.cvtColor(img_post, cv2.COLOR_BGR2RGB)
        entrada  = np.concatenate([img_pre, img_post], axis=2)

        etiquetas = json.loads((lbl_dir / f"{base}_post_disaster.json").read_text())
        alto, ancho = img_post.shape[:2]
        mascara = np.full((alto, ancho), fill_value=255, dtype=np.int64)

        for feat in etiquetas.get("features", {}).get("xy", []):
            clase = DANO_A_CLASE.get(feat["properties"].get("subtype", "un-classified"), 4)
            wkt = feat.get("wkt", "")
            if wkt:
                m = wkt_a_mascara(wkt, ancho, alto)
                mascara[m == 1] = clase

        if self.transformacion:
            aug = self.transformacion(image=entrada, mask=mascara)
            entrada, mascara = aug["image"], aug["mask"]

        return entrada.float() / 255.0, mascara.long()


def cargar_pares(img_dir, lbl_dir):
    if not img_dir.exists():
        return []
    return [
        (f.stem.replace("_post_disaster", ""), img_dir, lbl_dir)
        for f in img_dir.glob("*_post_disaster.png")
        if (img_dir / f.stem.replace("post", "pre")).with_suffix(".png").exists()
        and (lbl_dir / f.stem).with_suffix(".json").exists()
    ]


pares_t1 = cargar_pares(XBD_IMAGES_T1, XBD_LABELS_T1)
pares_t3 = cargar_pares(XBD_IMAGES_T3, XBD_LABELS_T3)
pares = sorted(pares_t1 + pares_t3, key=lambda x: x[0])
print(f"Pares tier1: {len(pares_t1)} | tier3: {len(pares_t3)} | Total: {len(pares)}")

transform_train = A.Compose([
    A.RandomCrop(TAMANO, TAMANO),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.2),
    ToTensorV2(),
])
transform_val = A.Compose([A.CenterCrop(TAMANO, TAMANO), ToTensorV2()])

n_val   = max(1, int(len(pares) * 0.2))
n_train = len(pares) - n_val
ds_train = XBDDataset(pares[:n_train], transform_train)
ds_val   = XBDDataset(pares[n_train:], transform_val)

BATCH = 8
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
print(f"Train: {len(ds_train)} | Val: {len(ds_val)} | Batch: {BATCH}")


## 4. Crear el modelo

In [ ]:
import segmentation_models_pytorch as smp

modelo = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=6,        # 3 canales pre + 3 canales post
    classes=N_CLASES,
    activation=None,
)
modelo = modelo.to(dispositivo)
n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"✅ Modelo U-Net ResNet34: {n_params:,} parámetros")


## 5. Entrenamiento

El modelo se guarda en `/kaggle/working/modelo_danos.pt` cada vez que mejora el IoU.
Puedes interrumpir el entrenamiento cuando quieras — el mejor modelo guardado se conserva.


In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

# Diagnóstico inicial
print("🔍 Diagnóstico del primer batch:")
for img_check, mask_check in dl_train:
    vals = torch.unique(mask_check).tolist()
    pct_255 = (mask_check == 255).float().mean().item() * 100
    pct_0   = (mask_check == 0).float().mean().item()   * 100
    print(f"   Valores en máscaras: {vals}")
    print(f"   Fondo ignorado (255): {pct_255:.1f}%")
    print(f"   Sin daño (0):         {pct_0:.1f}%")
    print(f"   Dañados (1-4):        {100-pct_255-pct_0:.1f}%")
    break

# Pérdida con pesos por clase e ignore_index para el fondo
PESOS_CLASE = torch.tensor([0.3, 2.0, 3.0, 4.0, 1.5]).to(dispositivo)
criterio = nn.CrossEntropyLoss(weight=PESOS_CLASE, ignore_index=255)

optimizador = AdamW(modelo.parameters(), lr=5e-5, weight_decay=1e-4)
N_EPOCAS = 30
scheduler = CosineAnnealingLR(optimizador, T_max=N_EPOCAS, eta_min=1e-7)


def calcular_iou(pred, target, n_clases=N_CLASES):
    pred_mask = pred.argmax(dim=1)
    ious = []
    for c in range(1, n_clases):          # solo clases dañadas (1-4)
        validos = target != 255
        p = (pred_mask == c) & validos
        t = (target == c)    & validos
        inter = (p & t).sum().float().item()
        union = (p | t).sum().float().item()
        if union > 0:
            ious.append(inter / union)
    return np.mean(ious) if ious else 0.0


mejor_iou = 0.0
historial  = {"train_loss": [], "val_loss": [], "val_iou": []}
print("\n🚀 Iniciando entrenamiento...\n")

for epoca in range(N_EPOCAS):
    modelo.train()
    perdida_train   = 0.0
    batches_validos = 0

    for imagenes, mascaras in dl_train:
        imagenes = imagenes.to(dispositivo)
        mascaras = mascaras.long().to(dispositivo)
        if (mascaras != 255).sum() == 0:
            continue
        optimizador.zero_grad()
        pred = modelo(imagenes)
        loss = criterio(pred, mascaras)
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        optimizador.step()
        perdida_train   += loss.item()
        batches_validos += 1
    perdida_train /= max(batches_validos, 1)

    modelo.eval()
    perdida_val = 0.0
    batches_val = 0
    ious        = []
    with torch.no_grad():
        for imagenes, mascaras in dl_val:
            imagenes = imagenes.to(dispositivo)
            mascaras = mascaras.long().to(dispositivo)
            if (mascaras != 255).sum() == 0:
                continue
            pred = modelo(imagenes)
            loss = criterio(pred, mascaras)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                perdida_val += loss.item()
                batches_val += 1
            ious.append(calcular_iou(pred, mascaras))
    perdida_val /= max(batches_val, 1)
    val_iou      = float(np.mean(ious)) if ious else 0.0

    scheduler.step()
    historial["train_loss"].append(perdida_train)
    historial["val_loss"].append(perdida_val)
    historial["val_iou"].append(val_iou)

    if val_iou > mejor_iou:
        mejor_iou = val_iou
        torch.save(modelo.state_dict(), "modelo_danos.pt")
        estrella = " ⭐ guardado"
    else:
        estrella = ""

    print(f"Época {epoca+1:02d}/{N_EPOCAS} | loss {perdida_train:.4f}/{perdida_val:.4f} | IoU {val_iou:.4f}{estrella}")

print(f"\n✅ Listo. Mejor IoU: {mejor_iou:.4f}")


## 6. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historial["train_loss"], label="Train")
ax1.plot(historial["val_loss"],   label="Val")
ax1.set_title("Pérdida (CrossEntropy)"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(historial["val_iou"], color="#1F9E7A")
ax2.set_title("IoU validación (clases dañadas)"); ax2.grid(alpha=0.3)
ax2.axhline(y=mejor_iou, color="#FA4531", linestyle="--", label=f"Mejor: {mejor_iou:.3f}")
ax2.legend()
plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=150)
plt.show()
print(f"Mejor IoU alcanzado: {mejor_iou:.4f}")


## 7. Descargar el modelo

Descarga `modelo_danos.pt` desde el panel **Output** (derecha) y cópialo en `backend/ia/`.

El backend lo detecta automáticamente al arrancar.

```
backend/
└── ia/
    └── modelo_danos.pt   ← aquí
```


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

# El modelo ya está en /kaggle/working/modelo_danos.pt
print(f"✅ Modelo listo en: /kaggle/working/modelo_danos.pt")
print(f"   Mejor IoU: {mejor_iou:.4f}")
print()
print("Descárgalo desde el panel Output (derecha) → modelo_danos.pt")
